In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

MY_SEED = 42  # TODO: 랜덤한 숫자
np.random.seed(MY_SEED)

#  Titanic 데이터를 사용
titanic = sns.load_dataset("titanic")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 1. 필수 과제

### 1-1. ML 문제 정의 (수업 §2)

코드를 짜기 전에, 이 문제를 ML 문제로 **정식화**합니다. 아래 빈칸을 채우세요.

- 우리는 이 데이터셋에서 타이타닉 탑승자의 생존 여부를 맞춰야합니다. **target(label)**은 어떤 column인가? → survived
- 이 데이터에는 label이 (있다). 따라서 학습 방식은 지도학습이다.
- target이 (범주형) 이므로, 이 문제는 (분류) 문제다. → 정답: 분류
- 학습 전에 데이터를 train/test로 나누는 이유를 **본인의 언어로** 한 문장: 학습에 쓴 데이터로 다시 평가하면 모델이 그 데이터를 이미 본 상태라 점수가 실제보다 부풀려지기 때문에, 한 번도 보여주지 않은 test로 따로 확인해야 맞는지 알 수 있다.


In [2]:
# 사용할 feature와 target 선택
features = ["pclass", "sex", "age", "fare"]
target = "survived"

df = titanic[features + [target]].copy()

# [빈칸] age의 결측치를 중앙값으로 채우는 이유 (평균이 아니라 중앙값을 쓰는 이유 포함):
# age는 갓난아기~고령까지 폭이 넓고 일부 고령 승객 같은 극단값 때문에
# 평균을 쓰면 값이 한쪽으로 쏠릴 수 있다. 중앙값은 극단값 영향을 덜 받아서
# 대표값으로 더 안전하다.
df["age"] = df["age"].fillna(df["age"].median())

# [빈칸] sex를 0/1 숫자로 바꾸는 이유 (모델 입장에서 설명):
# 모델은 결국 숫자 연산으로 학습하기 때문에 male/female 같은문자열을 그대로 넣을 수 없다.
# female이면 1, male이면 0으로 바꿔야 계산이 가능하다.
df["sex"] = (df["sex"] == "female").astype(int)

df = df.dropna()

X = df[features]
y = df[target]

# TODO: train/test를 8:2로 분할하세요. random_state=MY_SEED 사용
# [빈칸] stratify=y 옵션을 주는 이유: 원래 생존자 비율이 한쪽으로 치우쳐 있는데, 그냥 무작위로 나누면
# 우연히 train/test의 생존 비율이 크게 달라질 수 있다. stratify=y로 원래 비율을 양쪽에
# 그대로 유지해서 공정하게 비교할 수 있게 한다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=MY_SEED,
    stratify=y
)

print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"train 생존율: {y_train.mean():.3f}, test 생존율: {y_test.mean():.3f}")

train: (712, 4), test: (179, 4)
train 생존율: 0.383, test 생존율: 0.385


### 1-3. Gradient Descent 손계산 (수업 §3)

수업에서 "학습 = loss를 낮추는 방향으로 파라미터를 조금씩 갱신"이라고 했습니다. 이걸 **가장 작은 예제로 직접 계산**합니다.

**설정**: 데이터 3개 $(x, y) = (1,2), (2,4), (3,6)$, 모델 $\hat{y} = wx$ (절편 없음), loss는 MSE

$$L(w) = \frac{1}{3}\sum_{i=1}^{3}(y_i - wx_i)^2, \qquad \frac{\partial L}{\partial w} = -\frac{2}{3}\sum_{i=1}^{3} x_i(y_i - wx_i)$$

**문제**: $w_0 = 0$, learning rate $\eta = 0.1$에서 시작하여 **gradient descent를 3회 반복해 직접 계산**하고, **각 단계에서 $w$가 왜 그 방향으로, 왜 그만큼 움직였는지** 설명하세요. (계산기는 써도 되지만 코드로 먼저 답을 구하면 안 됩니다 — 아래 검증 셀은 손계산이 끝난 뒤에 실행)

| 반복 | 현재 $w$ | 각 데이터의 오차 $(y_i - wx_i)$ | gradient $\frac{\partial L}{\partial w}$ | 갱신된 $w$ |
|---|---|---|---|---|
| 1 | 0 | (2,4,6) | - 56/3(-18.667) | 1.867 |
| 2 | 1.867 | (0.133,0.267,0.400) | -1.244 | 1.991 |
| 3 | 1.991 | (0.009,0.018,0.027) | -0.083 | 1.999 |

**서술**: 매 반복마다 gradient의 크기가 어떻게 변했고, 그것이 $w$의 이동량과 어떤 관계인가? 이 과정이 언제, 왜 멈추게 될까? → gradient 크기가 -18.667 → -1.244 → -0.083으로 반복할수록 확 줄어든다. w가 실제 정답(이 데이터는 y=2x라서 w=2가 최적)에 가까워질수록 오차 $(y_i - wx_i)$가 작아지고, 그만큼 gradient도 작아지기 때문이다. 이동량은 $𝜂$ * gradient라서 gradient가 작아지면 w가 움직이는 폭도 같이 작아진다. 이 과정은 gradient가 0에 가까워질 때(즉 loss가 더 이상 줄지 않는 지점 근처) 사실상 멈추게 된다.


In [3]:
# ===== 손계산 검증용 셀 (표를 다 채운 뒤에 실행하세요) =====
# 주의: 위에서 만든 target y(Series)를 덮어쓰지 않도록 여기서는 gx, gy 라는 별도 변수를 쓴다.
gx = np.array([1, 2, 3])
gy = np.array([2, 4, 6])

w = 0.0
lr = 0.1

for step in range(1, 4):
    # TODO: 위 수식대로 gradient를 코드로 옮기세요
    grad = -2/len(gx) * np.sum(gx * (gy - w * gx))   # 힌트: -2/len(gx) * np.sum(...)

    # [빈칸] gradient의 '반대 방향'으로 이동하는 이유:
     # gradient는 loss가 가장 빠르게 증가하는 방향이기 때문에
     # loss 줄이려면 정반대 방향으로 가야한다.
    w = w - lr * grad

    loss = np.mean((gy - w * gx) ** 2)
    print(f"step {step}: grad = {grad:+.4f}, w = {w:.4f}, loss = {loss:.4f}")

# 손계산 결과와 출력이 일치하는지 확인하고, 다르면 어디서 틀렸는지 찾아 적으세요:
# 반복 2회차 중간에 계산 실수 나와서 다시 함, 최종은 거의 일치하는데 반올림 차이

step 1: grad = -18.6667, w = 1.8667, loss = 0.0830
step 2: grad = -1.2444, w = 1.9911, loss = 0.0004
step 3: grad = -0.0830, w = 1.9994, loss = 0.0000


### 1-4. 두 모델 학습과 비교

수업에서 "모든 알고리즘은 (f의 형태 / loss / 찾는 방법)에 대한 서로 다른 답"이라고 했습니다. 형태가 전혀 다른 두 모델을 **같은 데이터, 같은 조건**에서 학습시켜 비교합니다.

- **모델 A**: Logistic Regression (선형 + sigmoid)
- **모델 B**: Decision Tree (재귀 분할)


In [4]:
# 과정 확인 장치: train과 test 성능을 모두 기록합니다
results = {}

# TODO: 모델 A — LogisticRegression을 학습시키세요 (max_iter=1000)
model_a = LogisticRegression(max_iter=1000)
model_a.fit(X_train, y_train)

# TODO: 모델 B — DecisionTreeClassifier를 학습시키세요 (random_state=MY_SEED)
model_b = DecisionTreeClassifier(random_state=MY_SEED)
model_b.fit(X_train, y_train)

for name, model in [("Logistic Regression", model_a), ("Decision Tree", model_b)]:
    # [빈칸] train 정확도와 test 정확도를 '둘 다' 기록하는 이유 (수업 §3의 개념과 연결해서):
    # train acc만 보면 모델이 학습 데이터를 얼마나 잘 외웠는지만 알 수 있다.
    # train loss만 낮추는 건 쉽다고 한 것처럼, 처음 보는 데이터인 test acc까지 같이 봐야
    # 진짜 일반화가 됐는지 판단할 수 있다.
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = (train_acc, test_acc)
    print(f"{name:20s} | train acc: {train_acc:.4f} | test acc: {test_acc:.4f}")

Logistic Regression  | train acc: 0.7992 | test acc: 0.7877
Decision Tree        | train acc: 0.9803 | test acc: 0.8101


In [5]:
y_pred = model_a.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# sklearn의 confusion_matrix 배치: [[TN, FP], [FN, TP]]
TN, FP = cm[0]
FN, TP = cm[1]
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# TODO: 아래 4개 지표를 TP/TN/FP/FN '만으로' 직접 계산하세요 (sklearn 함수 사용 금지)
my_accuracy  = (TP + TN) / (TP + TN + FP + FN)
my_precision = TP / (TP + FP)   # [빈칸] precision의 분모에 들어가는 것과 그 의미:
# precision 분모 = TP+FP, 즉 양성이라고 예측한 것 전체이고 예측이 맞을 확률을 본다.
my_recall    = TP / (TP + FN)   # [빈칸] recall의 분모에 들어가는 것과 그 의미:
# recall 분모 = TP+FN, 즉 실제 양성 전체이고 진짜 양성을 놓치지 않고 잡은 비율을 본다.
my_f1        = 2 * my_precision * my_recall / (my_precision + my_recall)   # 힌트: precision과 recall의 조화평균

# 검증: sklearn과 비교 (통과하지 못하면 수식을 다시 확인)
assert np.isclose(my_accuracy,  accuracy_score(y_test, y_pred))
assert np.isclose(my_precision, precision_score(y_test, y_pred))
assert np.isclose(my_recall,    recall_score(y_test, y_pred))
assert np.isclose(my_f1,        f1_score(y_test, y_pred))
print("✅ 4개 지표 모두 sklearn과 일치")

Confusion Matrix:
 [[94 16]
 [22 47]]
TP=47, TN=94, FP=16, FN=22
✅ 4개 지표 모두 sklearn과 일치


### 1-6. 결과 해석 (필수 서술)

아래 4개 질문에 **본인의 실험 결과 수치를 근거로** 답하세요. (수치 없이 일반론만 쓰면 감점)

1. **왜 이러한 결과가 나왔는가?** — 두 모델의 test 정확도는 각각 얼마였고, 이 데이터에서 그 정도 성능이 나온 이유를 feature 관점에서 추측하면? → `MY_SEED=42로 고정한 결과 Logistic Regression은 train 0.7992 / test 0.7877, Decision Tree는 train 0.9803 / test 0.8101이 나왔다. 결과를 통해 pclass, sex, age, fare 네개의 feature만으로도 생존 여부를 설명할 정보가 충분히 담겨 있다는 것으로 이해할 수 있다. 특히 Decision Tree가 Logistic Regression보다 test acc가 오히려 더 높게 나왔는데, 아마 3등급 남성 중에서도 나이가 어리면 생존율이 다르거나 요금이 비싼 승객일수록 등급과 무관하게 생존율이 높은 것과 같은, 변수들끼리 얽혀서 나타나는 비선형적인 패턴이 이 데이터에 존재하기 때문일 것이다. Logistic Regression은 직선으로만 경계를 긋기 때문에 이런 상호작용을 잡아내기 어렵고, Decision Tree는 조건을 재귀적으로 보기에 이런 패턴을 자연스럽게 반영할 수 있다.`
2. **두 모델의 성능 차이는 어디에서 발생했는가?** — train acc와 test acc의 '차이'가 두 모델에서 어떻게 달랐는가? 이를 모델 구조(선형 경계 vs 재귀 분할)의 관점에서 해석하면? → `train-test 격차가 Logistic Regression은 0.7992→0.7877로 0.0115밖에 안 나는데, Decision Tree는 0.9803→0.8101로 0.17이나 벌어졌다. 격차만 보면 Decision Tree가 훨씬 더 과적합된 모델처럼 보이지만, 정작 test 성능 자체는 Decision Tree가 여전히 더 높다. 즉 depth 제한 없는 tree는 train 데이터의 노이즈까지 끌어안는 대가로 격차는 크게 벌어지지만, 그만큼 확보한 표현력 덕분에 선형 모델이 못 잡는 패턴까지 잡아내서 절대 성능은 오히려 앞선다는 걸 확인할 수 있었다. 즉 train test 격차가 크다 = 그 모델이 더 나쁘다는 걸 의미하는 건 아니라는 걸 확인할 수 있었다.`
3. **실험 조건을 변경하면 결과가 어떻게 달라지는가?** — `MY_SEED`를 다른 값으로 바꿔 1-2 ~ 1-4를 다시 실행해 보고, 어떤 수치가 얼마나 흔들렸는지 기록. 이 흔들림을 줄이는 방법으로 수업에서 배운 것은? → `아래 셀에서 MY_SEED=40으로 바꾸어 돌렸다. Logistic Regression의 test acc는 똑같이 0.7877로 똑같이 나왔고, Decision Tree도 0.7989→0.8101로 큰 폭은 아니지만 조금 움직였다. 반면 train-test 격차는 모두 Logistic Regression은 거의 0에 가깝고 Decision Tree는 약 0.18 근처로 일관되게 크다. 즉 어떤 데이터가 train/test로 나뉘는지와 무관하게, Decision Tree가 더 과적합 경향이 크다는 패턴 자체는 계속 유지됐다.`
4. **예상과 다른 결과가 나왔다면 그 원인은 무엇인가?** — 과제를 시작하기 전 예상과 달랐던 지점 하나와, 그 원인에 대한 본인의 가설: → `과제 전에는 depth 제한 없는 Decision Tree가 train 데이터를 거의 다 외우다시피 하니까 test에서는 Logistic Regression한테 밀릴 거라 예상했다. 그런데 seed 40, 42 두 번 다 Decision Tree가 test acc에서 더 높게 나와서 의외였다. train-test 격차가 큰 것과 실제 test 성능이 낮은 것은 별개라는 걸 알게 됐다.`


In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

MY_SEED = 40  # TODO: 랜덤한 숫자
np.random.seed(MY_SEED)

#  Titanic 데이터를 사용
titanic = sns.load_dataset("titanic")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [7]:
# 사용할 feature와 target 선택
features = ["pclass", "sex", "age", "fare"]
target = "survived"

df = titanic[features + [target]].copy()

# [빈칸] age의 결측치를 중앙값으로 채우는 이유 (평균이 아니라 중앙값을 쓰는 이유 포함):
# age는 갓난아기~고령까지 폭이 넓고 일부 고령 승객 같은 극단값 때문에
# 평균을 쓰면 값이 한쪽으로 쏠릴 수 있다. 중앙값은 극단값 영향을 덜 받아서
# 대표값으로 더 안전하다.
df["age"] = df["age"].fillna(df["age"].median())

# [빈칸] sex를 0/1 숫자로 바꾸는 이유 (모델 입장에서 설명):
# 모델은 결국 숫자 연산으로 학습하기 때문에 male/female 같은문자열을 그대로 넣을 수 없다.
# female이면 1, male이면 0으로 바꿔야 계산이 가능하다.
df["sex"] = (df["sex"] == "female").astype(int)

df = df.dropna()

X = df[features]
y = df[target]

# TODO: train/test를 8:2로 분할하세요. random_state=MY_SEED 사용
# [빈칸] stratify=y 옵션을 주는 이유: 원래 생존자 비율이 한쪽으로 치우쳐 있는데, 그냥 무작위로 나누면
# 우연히 train/test의 생존 비율이 크게 달라질 수 있다. stratify=y로 원래 비율을 양쪽에
# 그대로 유지해서 공정하게 비교할 수 있게 한다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=MY_SEED,
    stratify=y
)

print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"train 생존율: {y_train.mean():.3f}, test 생존율: {y_test.mean():.3f}")

train: (712, 4), test: (179, 4)
train 생존율: 0.383, test 생존율: 0.385


In [8]:
# ===== 손계산 검증용 셀 (표를 다 채운 뒤에 실행하세요) =====
# 주의: 위에서 만든 target y(Series)를 덮어쓰지 않도록 여기서는 gx, gy 라는 별도 변수를 쓴다.
gx = np.array([1, 2, 3])
gy = np.array([2, 4, 6])

w = 0.0
lr = 0.1

for step in range(1, 4):
    # TODO: 위 수식대로 gradient를 코드로 옮기세요
    grad = -2/len(gx) * np.sum(gx * (gy - w * gx))   # 힌트: -2/len(gx) * np.sum(...)

    # [빈칸] gradient의 '반대 방향'으로 이동하는 이유:
     # gradient는 loss가 가장 빠르게 증가하는 방향이기 때문에
     # loss 줄이려면 정반대 방향으로 가야한다.
    w = w - lr * grad

    loss = np.mean((gy - w * gx) ** 2)
    print(f"step {step}: grad = {grad:+.4f}, w = {w:.4f}, loss = {loss:.4f}")

# 손계산 결과와 출력이 일치하는지 확인하고, 다르면 어디서 틀렸는지 찾아 적으세요:
# 반복 2회차 중간에 계산 실수 나와서 다시 함, 최종은 거의 일치하는데 반올림 차이

step 1: grad = -18.6667, w = 1.8667, loss = 0.0830
step 2: grad = -1.2444, w = 1.9911, loss = 0.0004
step 3: grad = -0.0830, w = 1.9994, loss = 0.0000


In [9]:
# 과정 확인 장치: train과 test 성능을 모두 기록합니다
results = {}

# TODO: 모델 A — LogisticRegression을 학습시키세요 (max_iter=1000)
model_a = LogisticRegression(max_iter=1000)
model_a.fit(X_train, y_train)

# TODO: 모델 B — DecisionTreeClassifier를 학습시키세요 (random_state=MY_SEED)
model_b = DecisionTreeClassifier(random_state=MY_SEED)
model_b.fit(X_train, y_train)

for name, model in [("Logistic Regression", model_a), ("Decision Tree", model_b)]:
    # [빈칸] train 정확도와 test 정확도를 '둘 다' 기록하는 이유 (수업 §3의 개념과 연결해서):
    # train acc만 보면 모델이 학습 데이터를 얼마나 잘 외웠는지만 알 수 있다.
    # train loss만 낮추는 건 쉽다고 한 것처럼, 처음 보는 데이터인 test acc까지 같이 봐야
    # 진짜 일반화가 됐는지 판단할 수 있다.
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = (train_acc, test_acc)
    print(f"{name:20s} | train acc: {train_acc:.4f} | test acc: {test_acc:.4f}")

Logistic Regression  | train acc: 0.7809 | test acc: 0.7877
Decision Tree        | train acc: 0.9775 | test acc: 0.7989


In [10]:
y_pred = model_a.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# sklearn의 confusion_matrix 배치: [[TN, FP], [FN, TP]]
TN, FP = cm[0]
FN, TP = cm[1]
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# TODO: 아래 4개 지표를 TP/TN/FP/FN '만으로' 직접 계산하세요 (sklearn 함수 사용 금지)
my_accuracy  = (TP + TN) / (TP + TN + FP + FN)
my_precision = TP / (TP + FP)   # [빈칸] precision의 분모에 들어가는 것과 그 의미:
# precision 분모 = TP+FP, 즉 양성이라고 예측한 것 전체이고 예측이 맞을 확률을 본다.
my_recall    = TP / (TP + FN)   # [빈칸] recall의 분모에 들어가는 것과 그 의미:
# recall 분모 = TP+FN, 즉 실제 양성 전체이고 진짜 양성을 놓치지 않고 잡은 비율을 본다.
my_f1        = 2 * my_precision * my_recall / (my_precision + my_recall)   # 힌트: precision과 recall의 조화평균

# 검증: sklearn과 비교 (통과하지 못하면 수식을 다시 확인)
assert np.isclose(my_accuracy,  accuracy_score(y_test, y_pred))
assert np.isclose(my_precision, precision_score(y_test, y_pred))
assert np.isclose(my_recall,    recall_score(y_test, y_pred))
assert np.isclose(my_f1,        f1_score(y_test, y_pred))
print("✅ 4개 지표 모두 sklearn과 일치")

Confusion Matrix:
 [[86 24]
 [14 55]]
TP=55, TN=86, FP=24, FN=14
✅ 4개 지표 모두 sklearn과 일치


## 3. 생성형AI 활용 방법

문제를 풀면서 GPT, Claude 등 생성형 AI를 **어디에, 어떻게** 썼는지 기록하는 곳입니다. (사용하지 않았다면 "사용하지 않음"이라고 적으세요)

- **활용 방법**: 어떤 문제에서, 어떤 목적으로(개념 질문 / 에러 해결 / 코드 초안 등) 사용했는지 → 손풀이 문제에서 3회차 부분 값이 코드 값과 차이가 나서 어느 부분에서 틀렸는지 빠르게 알기 위해 AI한테 질문. 계산 실수 원인을 잡아냄.
- **AI 답변 중 그대로 쓰지 않고 직접 수정·검증한 부분**: → `______`
- **대화 내역 붙여넣기**: (아래에 전체 대화를 붙여넣으세요)

```
(대화 내역) 캡쳐한 풀이에서 틀린 부분을 찾아줘.
```

## 4. 회고

- **가장 어려웠던 부분**과 그것을 어떻게 해결했는지 (또는 아직 해결하지 못했는지, 없다면 없음이라고 적어도 됨): → `세션을 통해 얻어가는 것이 많아 전체적인 내용이 비전공자인 나에게 쉬운 편은 아니지만, 느리게 학습하며 어려운 부분들을 해결해 나가고 있다.`
- 이번 과제 내용 중 기술블로그 '과제 복습' 섹션에 정리할 핵심 1가지: → `이번 과제 하면서 제일 기억에 남은것은 train-test 격차(과적합 정도)가 크다고 무조건 그 모델이 나쁜 게 아니라는 점이다. Decision Tree는 depth 제한을 안 줬더니 train acc가 0.98까지 올라가서 딱 봐도 과적합인데, 정작 test acc는 오히려 Logistic Regression보다 계속 높게 나왔다. 격차가 크다 = 나쁜 모델이라고 단순하게 생각하면 안 되고, 절대적인 test 성능이랑 같이 봐야 한다는 것이 핵심이다.`
